# vLLM Performance Analysis

This notebook analyzes performance metrics from the load test.
Run `make perf` first to generate `metrics.csv`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["figure.dpi"] = 100

In [ ]:
df = pd.read_csv("metrics.csv")
print(f"Loaded {len(df)} records")
df.head(10)

## 1. Time-to-First-Token (TTFT) vs Concurrency

TTFT measures how quickly the server begins generating tokens.
Higher concurrency should increase TTFT as the server batches more requests.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, prompt_type in zip(axes, ["short", "long"]):
    subset = df[(df["prompt_type"] == prompt_type) & (df["stop_setting"] == "none")]
    stats = subset.groupby("concurrency")["ttft_ms"].agg(["median", lambda x: np.percentile(x, 95)])
    stats.columns = ["P50", "P95"]
    stats.plot(kind="bar", ax=ax)
    ax.set_title(f"TTFT vs Concurrency ({prompt_type} prompts)")
    ax.set_xlabel("Concurrency")
    ax.set_ylabel("TTFT (ms)")
    ax.legend()

plt.tight_layout()
plt.savefig("ttft_vs_concurrency.png", bbox_inches="tight")
plt.show()

## 2. Throughput (Tokens/sec) vs Concurrency

Aggregate throughput should increase with concurrency due to continuous batching,
up to the point where the GPU is saturated.

In [ ]:
no_stop = df[df["stop_setting"] == "none"].copy()

agg_throughput = (
    no_stop.groupby(["concurrency", "prompt_type"])
    .agg(total_tokens=("tokens_generated", "sum"),
         total_time_ms=("total_latency_ms", "max"))
    .reset_index()
)
agg_throughput["throughput"] = agg_throughput["total_tokens"] / (agg_throughput["total_time_ms"] / 1000)

fig, ax = plt.subplots(figsize=(10, 6))
for ptype in ["short", "long"]:
    sub = agg_throughput[agg_throughput["prompt_type"] == ptype]
    ax.plot(sub["concurrency"], sub["throughput"], marker="o", label=ptype)

ax.set_xlabel("Concurrency")
ax.set_ylabel("Aggregate Throughput (tokens/sec)")
ax.set_title("Throughput vs Concurrency")
ax.legend()
ax.set_xscale("log", base=2)
plt.tight_layout()
plt.savefig("throughput_vs_concurrency.png", bbox_inches="tight")
plt.show()

## 3. Latency Distribution (Box Plots)

Box plots show the spread of end-to-end latency at each concurrency level.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, prompt_type in zip(axes, ["short", "long"]):
    subset = df[(df["prompt_type"] == prompt_type) & (df["stop_setting"] == "none")]
    sns.boxplot(data=subset, x="concurrency", y="total_latency_ms", ax=ax)
    ax.set_title(f"Latency Distribution ({prompt_type} prompts)")
    ax.set_xlabel("Concurrency")
    ax.set_ylabel("Total Latency (ms)")

plt.tight_layout()
plt.savefig("latency_distribution.png", bbox_inches="tight")
plt.show()

## 4. Effect of Stop Sequences

Comparing how different stop-sequence settings affect latency and tokens generated.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Tokens generated by stop setting
sns.barplot(data=df, x="stop_setting", y="tokens_generated", hue="prompt_type", ax=axes[0])
axes[0].set_title("Tokens Generated by Stop Setting")
axes[0].set_ylabel("Tokens Generated")

# Latency by stop setting
sns.barplot(data=df, x="stop_setting", y="total_latency_ms", hue="prompt_type", ax=axes[1])
axes[1].set_title("Latency by Stop Setting")
axes[1].set_ylabel("Total Latency (ms)")

plt.tight_layout()
plt.savefig("stop_sequence_effect.png", bbox_inches="tight")
plt.show()

## 5. Summary Statistics

Percentile breakdown across all configurations.

In [ ]:
summary = (
    df.groupby(["concurrency", "prompt_type", "stop_setting"])
    .agg(
        count=("ttft_ms", "count"),
        ttft_p50=("ttft_ms", "median"),
        ttft_p95=("ttft_ms", lambda x: np.percentile(x, 95)),
        ttft_p99=("ttft_ms", lambda x: np.percentile(x, 99)),
        latency_p50=("total_latency_ms", "median"),
        latency_p95=("total_latency_ms", lambda x: np.percentile(x, 95)),
        latency_p99=("total_latency_ms", lambda x: np.percentile(x, 99)),
        avg_tpot=("tpot", "mean"),
        avg_tokens=("tokens_generated", "mean"),
    )
    .round(2)
)

summary

## Commentary

**Key observations:**

1. **TTFT scales with concurrency**: As more requests are processed simultaneously, the time to receive the first token increases because the model must compute prefill for a larger batch.

2. **Throughput benefits from batching**: Continuous batching in vLLM allows aggregate throughput to increase with concurrency, as the GPU processes multiple sequences in parallel during the decode phase.

3. **Long vs short prompts**: Long prompts have higher TTFT due to longer prefill computation, but per-token decode speed remains similar since the decode phase is memory-bandwidth bound regardless of context length.

4. **Stop sequences reduce latency**: Enabling stop sequences (e.g., `\n\n`) terminates generation early, reducing both total latency and tokens generated. This is a useful optimization when full-length generation is not needed.

5. **Tail latency**: P99 latency is significantly higher than P50, which is typical in batched serving systems where some requests wait for longer-running sequences in the same batch to complete.